# PyTorch Neural Networks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caam37830-sci-comp/notes/blob/main/08_data/pytorch.ipynb)

[PyTorch](https://pytorch.org/) is a Python package for defining and training neural networks.  Neural networks and deep learning have been a hot topic for several years, and are the tools underlying many state-of-the art machine learning tasks.  There are many industrial applications (e.g. at your favorite or least favorite companies in Silicon Valley), but also many scientific applications including
* Processing data in particle detectors
* Seismic imaging / medical imaging
* Accelerating simulations of physical phenomena
* ...

A (deep) feed-forward neural network is the composition of functions
\begin{equation}
f_N(x; w_N, b_N) \circ f_{N-1}(x; w_{N-1}, b_{N-1}) \circ \dots f_0(x; w_0, b_0)
\end{equation}
where each $f_i(x; w_i, b_i)$ is a (non-linear) function with learnable parameters $w_i, b_i$.  There are many choices for what the exact function is.  A common and simple one to describe is an (affine) linear transformation followed by a non-linearity.
\begin{equation}
f_i(x; w_i, b_i) = (w_i \cdot x + b_i)_+
\end{equation}

where $w_i \cdot x$ is matrix-vector multiplication, and $(\cdot)_+$ is the ReLU operation (Rectified Linear Unit)
\begin{equation}
x_+ = \begin{cases}
x & x > 0\\
0 & x \le 0
\end{cases}
\end{equation}

If you take the composition of several functions like this, you have a multilayer perceptron (MLP).


## Deep Learning Libraries

The most common deep learning libraries for Python are

- PyTorch
- JAX
- TensorFlow, with its higher level interface Keras

All of them provide arrays that can live on a GPU, automatic differentiation, and building blocks for neural networks.  PyTorch is the most widely used in research, and it is the one we'll use.  Since version 3, Keras can also run on top of PyTorch or JAX.

### Main features that any deep learning library should provide

No matter what library or language you use, the main features provided by a deep learning library are
1. Use the GPU to speed up computation
2. Ability to do automatic differentiation
3. Useful library functions for common architectures and optimization algorithms

### PyTorch
We will look at all of the above in pytorch.
The best way to think about pytorch is that it's numpy + GPU + autograd.  PyTorch tensors, its version of NumPy arrays, were introduced in [Linear Algebra in PyTorch](../02_linear_algebra/pytorch.ipynb).

You can install it, together with the `torchvision` package we use below, with

```bash
(pycourse) $ conda install pytorch torchvision
```

or with `pip install torch torchvision` (see the [installation page](https://pytorch.org/get-started/locally/) for GPU versions).

Alternatively (and recommended), run this notebook in Google Colab-- it provides an environment with all of the PyTorch dependencies plus a GPU free of charge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision.datasets import MNIST

torch.__version__

## Automatic Differentiation

[Automatic differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation) is different from numerical differentiation, which requires a choice of step size, and symbolic differentiation which creates a single expression for a derivative.  Instead it performs chain rule repeatedly.

PyTorch uses dynamic computation graphs to compute the gradients of the parameters.

In [ ]:
x = torch.tensor([2.0])
m = torch.tensor([5.0], requires_grad = True)
c = torch.tensor([2.0], requires_grad = True)

In [ ]:
y = m*x + c
y

Define an error for your function

In [ ]:
loss = torch.norm( y - 13)
loss

In [ ]:
m.grad

Calling `x.backward()` on a scalar tensor `x` forces pytorch to compute the gradient of `x` with respect to all the tensors used to compute `x` which had the `requires_grad` flag set to `True`. The computed gradient will be stored in the `.grad` property of those tensors

In [ ]:
loss.backward()

In [ ]:
m.grad

In [ ]:
c.grad

In [ ]:
with torch.no_grad():
    m -= 0.01 * m.grad
    c -= 0.3 * c.grad

In [ ]:
m,c

In [ ]:
m.grad, c.grad

In [ ]:
m.grad.zero_()
c.grad.zero_()

m.grad, c.grad

In [ ]:
y = m*x + c
y

In [ ]:
loss = torch.norm( y - 13)
loss

In [ ]:
loss.backward()
m.grad, c.grad

### Making it more compact

In [ ]:
def model_fn(x,m,c):
    return m*x + c

def loss_fn(y,yt):
    return torch.norm(y-yt)

In [ ]:
m = torch.tensor([5.0], requires_grad = True)
c = torch.tensor([2.0], requires_grad = True)

In [ ]:
x = torch.tensor([2.0])
yt = torch.tensor([13.0])

In [ ]:
y = model_fn(x,m,c)
loss = loss_fn(y,yt)
loss.backward()
with torch.no_grad():
    m -= 0.05 * m.grad
    c -= 0.05 * c.grad
m.grad.zero_()
c.grad.zero_()

print( f" m = {m}\n c = {c}\n y = {y}\n loss = {loss}")
#note that 'loss' indicates the loss for the previous m,c values

Here's an explicit loop, fitting a line to 100 noisy samples of $y = 3x + 1$:

In [ ]:
x = torch.linspace(-1, 1, 100)
yt = 3 * x + 1 + 0.1 * torch.randn(100)

m = torch.tensor([5.0], requires_grad = True)
c = torch.tensor([2.0], requires_grad = True)
losses = []

for i in range(100):
    y = model_fn(x,m,c)
    loss = loss_fn(y,yt)
    loss.backward()
    with torch.no_grad():
        m -= 0.01 * m.grad
        c -= 0.01 * c.grad
    m.grad.zero_()
    c.grad.zero_()

    losses+=[loss.item()]

print(f"m = {m.item():.3f}, c = {c.item():.3f}")
plt.plot(losses);

The loss levels off at about $0.1 \sqrt{100} = 1$, the size of the noise, and $m, c$ are close to $3, 1$.

## Using Library functions

The subpackage `torch.nn` provides an object-oriented library of functions that can be composed together.


In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(5, 5), # 5 x 5 matrix
    torch.nn.ReLU(),       # ReLU nonlinearity
    torch.nn.Linear(5, 5), # 5 x 5 matrix
)

In [ ]:
list(model.parameters())

In [ ]:
loss_fn = torch.nn.MSELoss(reduction='sum')

In this case, we'll just fit the model to random data.

In [ ]:
x = torch.randn(100,5)
yt = torch.randn(100,5)
losses = []

Optimizers in `torch.optim` implement a variety of optimization strategies.  Almost all are based on gradient descent, since forming Hessians is prohibitive.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

In [ ]:
for i in range(100):
    y = model(x)
    loss = loss_fn(y,yt)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    losses+=[loss.item()]

plt.plot(losses);

## MNIST Example

The [MNIST data set](https://en.wikipedia.org/wiki/MNIST_database) consists of a collection of handwritten digits (0-9).  Our goal is to train a neural net which will classify the image of each digit as the correct digit

We load it with the `torchvision` package, which provides a variety of computer vision functionality for PyTorch, including common data sets.

In [ ]:
import torchvision

In [ ]:
from torchvision.datasets import MNIST

In [ ]:
data = MNIST(".", download=True)                    # 60,000 training images
test_data = MNIST(".", train=False, download=True)  # 10,000 test images

In [ ]:
len(data)

In [ ]:
i = torch.randint(len(data), ()).item()  # a random training image
img, y = data[i]
print(y)
img

In [ ]:
data.data[2].shape

In [ ]:
data.targets[2]

### MNIST Training

Each training step uses a random sample of 1000 training images.  We flatten each $28 \times 28$ image into a vector of length 784 and scale the pixel values to $[0, 1]$.

In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(784, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 10),
)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
sample = torch.randint(len(data), (1000,))
x = data.data[sample].reshape(1000,-1).float()/255
yt = data.targets[sample]

In [ ]:
x.shape,yt.shape

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
losses = []

In [ ]:
for i in range(100):

    sample = torch.randint(len(data), (1000,))
    x = data.data[sample].reshape(1000,-1).float()/255
    yt = data.targets[sample]

    y = model(x)
    loss = loss_fn(y,yt)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    losses+=[loss.item()]
plt.plot(losses);

We measure the accuracy on the 10,000 test images, which were not used for training.

In [ ]:
x_test = test_data.data.reshape(len(test_data),-1).float()/255
y_test = test_data.targets

In [ ]:
with torch.no_grad():
    y_pred = model(x_test)

In [ ]:
accuracy = (y_pred.argmax(dim=1) == y_test).float().mean().item()
print(f"Accuracy = {accuracy:.3f}")

### Credits

This notebook was adapted from a notebook from [CME 193 at Stanford](https://github.com/icme/cme193)